# 🔥 OPTUNA 100 Trials + Pruning + CatBoost (Colab Ready)

This notebook is organized for Google Colab execution.

## Pipeline
1. Install Libraries
2. Imports
3. Feature Engineering
4. Data Preparation
5. Optuna Optimization
6. Final Training
7. Submission


In [ ]:
# ==========================================================
# Install (Colab)
# ==========================================================
!pip install -q optuna catboost optuna-integration


In [ ]:
# ==========================================================
# Imports
# ==========================================================
import optuna
from optuna.integration import CatBoostPruningCallback
from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd


In [ ]:
# ==========================================================
# Feature Engineering
# ==========================================================
def make_features_full(df, formula_pred=None):
    df = df.copy()

    df['Weight_kg']     = df['Weight(lb)'] * 0.453592
    df['Height_inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['BMI']           = (df['Weight(lb)'] / df['Height_inches']**2) * 703
    df['Gender_enc']    = (df['Gender'] == 'M').astype(int)

    ws_map = {'Normal Weight':0, 'Overweight':1, 'Obese':2}
    df['WeightStatus_enc'] = df['Weight_Status'].map(ws_map).fillna(0)

    df['Dur_BPM']        = df['Exercise_Duration'] * df['BPM']
    df['Dur_Temp']       = df['Exercise_Duration'] * df['Body_Temperature(F)']
    df['Dur_BPM_Temp']   = df['Exercise_Duration'] * df['BPM'] * df['Body_Temperature(F)']
    df['HR_per_Weight']  = df['BPM'] / df['Weight_kg']
    df['Temp_diff']      = df['Body_Temperature(F)'] - 98.6
    df['Age_Weight']     = df['Age'] * df['Weight_kg']
    df['Age_Duration']   = df['Age'] * df['Exercise_Duration']
    df['BPM_squared']    = df['BPM']**2
    df['Dur_squared']    = df['Exercise_Duration']**2

    if formula_pred is not None:
        df['Formula_Pred'] = formula_pred

    return df


In [ ]:
# ==========================================================
# Data Preparation (Fill with your data)
# ==========================================================

# Example:
# train = pd.read_csv('train.csv')
# test  = pd.read_csv('test.csv')

# formula_pred_train = ...
# formula_test_pred  = ...

# formula_test_pred = formula_predict(params_best, hr_t, wt_t, age_t, dur_t, temp_t, male_t)

# X      = make_features_full(train, formula_pred_train)
# X_test = make_features_full(test,  formula_test_pred)

# y = train['Calories_Burned'].values

print('👉 Please prepare train/test before running Optuna')


In [ ]:
# ==========================================================
# Optuna Objective
# ==========================================================
def objective(trial):

    params = {
        "iterations": 8000,
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05),
        "depth": trial.suggest_int("depth", 6, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 30),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "random_strength": trial.suggest_float("random_strength", 0, 10),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": False
    }

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in kf.split(X):

        model = CatBoostRegressor(**params)

        model.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=(X.iloc[val_idx], y[val_idx]),
            early_stopping_rounds=400,
            callbacks=[CatBoostPruningCallback(trial, "RMSE")]
        )

        oof[val_idx] = model.predict(X.iloc[val_idx])

    rmse = np.sqrt(mean_squared_error(y, oof))
    return rmse


In [ ]:
# ==========================================================
# Run Optuna
# ==========================================================
print("🔥 OPTUNA 100 Trials 시작 (시간 오래 걸림)")

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=100)

print("\n🔥 최적 파라미터")
print(study.best_params)
print(f"🔥 Best RMSE: {study.best_value:.12f}")


In [ ]:
# ==========================================================
# Final Training
# ==========================================================
best_params = study.best_params
best_params.update({
    "iterations": 8000,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False
})

kf = KFold(n_splits=10, shuffle=True, random_state=42)

oof_pred  = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

print("\n🔥 Best Params로 10-Fold 최종 학습")

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):

    model = CatBoostRegressor(**best_params)

    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=(X.iloc[val_idx], y[val_idx]),
        early_stopping_rounds=400
    )

    oof_pred[val_idx] = model.predict(X.iloc[val_idx])
    test_pred += model.predict(X_test) / 10

    print(f"  Fold {fold+1}/10 완료")

oof_rmse     = np.sqrt(mean_squared_error(y, oof_pred))
rounded_rmse = np.sqrt(mean_squared_error(y, np.round(oof_pred)))

print("\n" + "="*60)
print("🔥🔥🔥 OPTUNA 100 TRIAL RESULT 🔥🔥🔥")
print("="*60)
print(f"OOF RMSE:        {oof_rmse:.15f}")
print(f"Rounded RMSE:    {rounded_rmse:.15f}")
print("="*60)


In [ ]:
# ==========================================================
# Submission
# ==========================================================
# sub = pd.read_csv('sample_submission.csv')

final_pred_int = np.clip(np.round(test_pred).astype(int), 1, 300)
sub['Calories_Burned'] = final_pred_int

sub.to_csv('submit_optuna_100.csv', index=False)

print("\n✅ submit_optuna_100.csv 저장 완료")
